# 05 · Joins — Combining Tables

Real questions span multiple tables. Joins connect them on matching keys.
- `INNER JOIN` — only matching rows in both tables
- `LEFT JOIN` — all rows from the left, matches from the right (or NULLs)
- multi-table joins
- self joins (a table joined to itself)
- `CROSS JOIN` (every combination)

In [ ]:
# ▶ Run this cell first. It loads JupySQL and connects to the SQLite database.
%load_ext sql
from sqlalchemy import create_engine
import os

# Works whether the notebook's working dir is the repo root or notebooks/
db_path = 'data/retail.db' if os.path.exists('data/retail.db') else '../data/retail.db'
engine = create_engine(f'sqlite:///{db_path}')

%config SqlMagic.autopandas = True      # results come back as pandas DataFrames
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = 0
%config SqlMagic.displaylimit = 100

%sql engine
print('Connected to', db_path)

## `INNER JOIN`
Products don't store the category *name*, only `category_id`. Join to
`categories` to get readable names. The `ON` clause says how rows match.

In [ ]:
%%sql
SELECT p.product_name, c.category_name, p.unit_price
FROM products AS p
INNER JOIN categories AS c ON p.category_id = c.category_id
ORDER BY c.category_name, p.product_name
LIMIT 10;

## Table aliases
`p` and `c` above are short aliases — they keep queries readable and are required when the same column name exists in both tables.

## Joining three tables
Which employee handled each order, and for which customer?

In [ ]:
%%sql
SELECT o.order_id,
       cu.first_name || ' ' || cu.last_name AS customer,
       e.first_name  || ' ' || e.last_name  AS employee,
       o.order_date
FROM orders AS o
JOIN customers AS cu ON o.customer_id = cu.customer_id
JOIN employees AS e  ON o.employee_id = e.employee_id
ORDER BY o.order_id
LIMIT 10;

## `LEFT JOIN` — keep unmatched left rows
Every product, plus its supplier name. Some products may have no supplier match;
`LEFT JOIN` keeps them with `NULL` on the supplier side.

In [ ]:
%%sql
SELECT p.product_name, s.supplier_name
FROM products AS p
LEFT JOIN suppliers AS s ON p.supplier_id = s.supplier_id
ORDER BY p.product_name
LIMIT 10;

### Finding rows with NO match
`LEFT JOIN ... WHERE right.key IS NULL` is the classic "anti-join". Which
customers have never placed an order?

In [ ]:
%%sql
SELECT cu.customer_id, cu.first_name, cu.last_name
FROM customers AS cu
LEFT JOIN orders AS o ON cu.customer_id = o.customer_id
WHERE o.order_id IS NULL;

## Joins + aggregation
The most powerful combination. Revenue per category (quantity × price, summed):

In [ ]:
%%sql
SELECT c.category_name,
       ROUND(SUM(oi.quantity * oi.unit_price), 2) AS revenue
FROM order_items AS oi
JOIN products   AS p ON oi.product_id = p.product_id
JOIN categories AS c ON p.category_id = c.category_id
GROUP BY c.category_name
ORDER BY revenue DESC;

## Self join
`employees.manager_id` points back into `employees`. Join the table to itself to
show each employee alongside their manager.

In [ ]:
%%sql
SELECT e.first_name || ' ' || e.last_name AS employee,
       e.title,
       m.first_name || ' ' || m.last_name AS manager
FROM employees AS e
LEFT JOIN employees AS m ON e.manager_id = m.employee_id
ORDER BY e.employee_id;

## Practice

**✏️ Exercise 1.** List each product with its supplier's country (product_name, country).

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
SELECT p.product_name, s.country
FROM products AS p
JOIN suppliers AS s ON p.supplier_id = s.supplier_id
ORDER BY p.product_name;

**✏️ Exercise 2.** Show every order with the customer's full name and the order status, for completed orders only.

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
SELECT o.order_id, cu.first_name || ' ' || cu.last_name AS customer, o.status
FROM orders AS o
JOIN customers AS cu ON o.customer_id = cu.customer_id
WHERE o.status = 'completed'
ORDER BY o.order_id;

**✏️ Exercise 3.** Compute total revenue per customer (join orders → order_items). Show the top 5 customers by revenue.

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
SELECT cu.first_name || ' ' || cu.last_name AS customer,
       ROUND(SUM(oi.quantity * oi.unit_price), 2) AS revenue
FROM customers AS cu
JOIN orders AS o      ON cu.customer_id = o.customer_id
JOIN order_items AS oi ON o.order_id = oi.order_id
GROUP BY cu.customer_id
ORDER BY revenue DESC
LIMIT 5;

### ✅ Recap
`INNER JOIN` keeps matches; `LEFT JOIN` keeps all left rows; anti-joins find
missing matches; self-joins relate a table to itself. Joins + `GROUP BY` answer
most business questions.

**Next:** `06_subqueries.ipynb`.